### Downscaling Algorithm Dev

In [1]:
import pandas as pd
from datetime import datetime

Create datasets with which to test algorithm

In [16]:
Tvals = [1,2,3,1,1,0,3,3]
Ttimes = pd.date_range(start='2025-01-01', end='2025-01-02 18:00', freq='6h')
T2data = pd.DataFrame({
    'Tmax': [5, 5],
    'Tmin': [0, 1]
}, index=pd.to_datetime(['2025-01-01', '2025-01-02']))

# Make variables that match the function
snotel_Tmax_K = T2data['Tmax']
snotel_Tmin_K = T2data['Tmin']
nldas_Tair_k = pd.Series(Tvals, index=Ttimes)

I need to know the index and the magnitude of the max and min values in order to adjust them.

In [17]:
nldas_max_idx = nldas_Tair_k.groupby(nldas_Tair_k.index.date).idxmax()
nldas_min_idx = nldas_Tair_k.groupby(nldas_Tair_k.index.date).idxmin()

In [18]:
nldas_max = nldas_Tair_k.loc[nldas_max_idx]
nldas_min = nldas_Tair_k.loc[nldas_min_idx]

Now I need the date of the max and min values in order to get the max and min values of the day from snotel data. There should always be one max and min per day, so I only need to find one set of dates. It is plausible that the snotel data could have some missing values, but I will worry about that later.

In [19]:
nldas_dates = nldas_max.index.normalize()
snotel_max = snotel_Tmax_K.loc[nldas_dates]
snotel_min = snotel_Tmin_K.loc[nldas_dates]

In [23]:
# Calculate the difference, always nldas - snotel so that the shift is consistent
diff_max = pd.Series(snotel_max.to_numpy() - nldas_max.to_numpy(), index = nldas_max.index)
diff_min = pd.Series(snotel_min.to_numpy() - nldas_min.to_numpy(), index = nldas_min.index)
snotel_diffs = pd.concat([diff_max, diff_min], axis=0)
snotel_diffs = snotel_diffs.sort_index()
snotel_diffs.name = 'snotel_diffs'

In [30]:
# Join with original nldas data
nldas_Tair_k.name = 'Tair'
nldas_Tair_k_df = pd.merge(nldas_Tair_k, snotel_diffs, how='left', left_index=True, right_index=True)

In [31]:
# If the first or last endpoints are missing, set to zero
if pd.isna(nldas_Tair_k_df['snotel_diffs'].iloc[0]):
    nldas_Tair_k_df.loc[nldas_Tair_k_df.index[0],'snotel_diffs'] = 0
if pd.isna(nldas_Tair_k_df['snotel_diffs'].iloc[-1]):
    nldas_Tair_k_df.loc[nldas_Tair_k_df.index[-1],'snotel_diffs'] = 0

# Interpolate the missing values in diffs
nldas_Tair_k_df['snotel_diffs'] = nldas_Tair_k_df['snotel_diffs'].interpolate()

In [32]:
nldas_Tair_k_df['Tair_corrected'] = nldas_Tair_k_df['Tair'] + nldas_Tair_k_df['snotel_diffs']

Consider the following:

In [ ]:
# Testing - This indicates that there will be a failure if one of the dates is not in the snotel data
# It might be more likely that there will be a date with a NAN
extended_dates = nldas_dates.append(pd.to_datetime(['2025-01-03']))
test = snotel_Tmax_K.loc[extended_dates]

KeyError: "[Timestamp('2025-01-03 00:00:00')] not in index"